# Capítulo 13 — Mejorar sin romper

**Formación Amazon Bedrock AgentCore · octubre 2026 · Caso de uso: Nova Assist (TiendaNova)**

> 🎬 **Escena.** Tenemos un prompt mejor… en teoría. ¿Lo cambiamos para todos los clientes un viernes por la tarde?

| | |
|---|---|
| **🧠 Componente agéntico** | Mejora continua y experimentación |
| **La necesidad** | Validar cambios de comportamiento con tráfico real y datos antes de promoverlos |
| **Lo que construimos** | **Configuration bundles** + **A/B testing** en el Gateway con evaluación online |
| **🏠 En local** | ☁️ No aplica: experimentar requiere tráfico real de clientes. |
| **☁️ Por qué AWS al salir a Internet** | En local, cambiar el prompt es redesplegar para todos. Los configuration bundles separan configuración y código, y el Gateway reparte el tráfico y mide con evaluación online antes de promover. |
| **En el ordenador sería…** | **Probar una actualización en unos pocos equipos** antes de instalarla en todos |
| **Requisitos del caso** | RNF-06 |
| **Prerrequisito · duración** | Capítulo 12 · 60 min |
| **⚠️ Dónde se rompe la analogía** | Una actualización en pocos equipos se prueba sobre máquinas; aquí se prueba **sobre conversaciones reales con clientes**. El muestreo y la evaluación online son obligatorios. |

In [ ]:
# Configuración común: ejecuta siempre esta celda primero
import os
from lab_utils import *

os.environ.setdefault("AWS_REGION", region())
cuaderno()

> 🎮 **¿Llegas tarde o algo se rompió?** La celda siguiente comprueba que tienes todo lo necesario. Si no, descomenta `ponerse_al_dia(13)`: carga la partida guardada al final del capítulo anterior o, si no existe, escribe el código de referencia y te dice qué recursos faltan y en qué capítulo se crean.

In [ ]:
requisitos(13)
# ponerse_al_dia(13)   # ← descomenta si llegas tarde o algo se rompió

## 🧠 La teoría: mejora continua

- Cambiar el comportamiento (prompt, modelo, herramientas) **con evidencia**, no con opiniones.
- **Separar configuración y código** para cambiar sin redesplegar.
- **Experimentos A/B:** repartir tráfico real entre variantes y decidir con significancia estadística.

> Este componente **no existe en local**: experimentar requiere tráfico real.

**Hipótesis de ejemplo:** "Un *system prompt* que pide confirmar el pedido antes de cada acción de escritura mejora `GoalSuccessRate` sin empeorar `Helpfulness`."

Variante alternativa: comparar dos inference profiles (doc §2).

> Las *flags* de estos comandos dependen de la versión de la CLI. Cada paso empieza mostrando la ayuda; complétalos con los valores de tu proyecto.

## 1. Preparación

In [ ]:
fijar_dependencia(VERSIONES["bedrock-agentcore"])   # incluye configuration bundles (>= 1.8.0)

Adapta `main.py` para leer el *system prompt* (y, si quieres, el `model_id`) del *configuration bundle* que llega en cada petición mediante cabeceras W3C *baggage*, usando el SDK de AgentCore. Sigue la guía *Run an A/B test with configuration bundles*. Mantén `prompts.py` como valor por defecto si no llega ningún bundle.

## 2. Configuration bundles: control y tratamiento

In [ ]:
!cd "{PROJECT}" && agentcore add config-bundle --help

In [ ]:
# Completa con la ayuda anterior: versión de control (prompt actual) y de tratamiento (prompt con confirmación)
# !cd "{PROJECT}" && agentcore add config-bundle ...
# !cd "{PROJECT}" && agentcore add config-bundle ...
# !cd "{PROJECT}" && agentcore deploy --yes

## 3. Runtime detrás del Gateway y lanzamiento del A/B

El A/B se reparte en el Gateway: publica el Runtime como target `http-runtime` y reutiliza la evaluación online del capítulo 12.

In [ ]:
!cd "{PROJECT}" && agentcore run ab-test --help

In [ ]:
# !cd "{PROJECT}" && agentcore run ab-test --online-eval nova_online_eval ...
# !cd "{PROJECT}" && agentcore view ab-test <ID>
# !cd "{PROJECT}" && agentcore promote ab-test -i <ID>     # escribe la variante ganadora en agentcore.json
# !cd "{PROJECT}" && agentcore deploy --yes

## 4. Recommendations (opcional)

Pide **Recommendations** sobre las trazas de producción para que AgentCore proponga mejoras del *system prompt* y de las descripciones de herramientas. Valida la propuesta con ***batch evaluations*** sobre el conjunto de prueba del capítulo 12 antes de lanzar otro A/B.

## Para pensar

- ¿Qué cambios merecen un A/B y cuáles basta con validar con *batch evaluations*?

## 🔎 Lo que acabamos de descubrir

Dirección insiste: *«¿Por qué se devuelven tantos robots S9? ¿Es cosa nuestra o del sector? ¿Qué hacen los demás? Quiero un informe el lunes.»* No es una conversación con un cliente: es una **investigación**, con tres trabajos distintos.

In [ ]:
cuaderno()
taximetro_cerrar(13)
guardar_partida(13)

---
➡️ **Siguiente capítulo:** `14_equipo_investigacion.ipynb` — *Del agente al equipo*